In [ ]:
"""
MODEL POPYTU BOTTOM-UP - stacje/huby ladowania EV w Polsce
============================================================

Struktura (identyczna dla obu segmentow):
    Popyt bazowy (EV) x Sklonnosc do ladowania publicznego x Udzial lokalizacji
    -> sesje/rok -> energia (kWh)/rok
"""

import pandas as pd
import numpy as np

# ============================================================
# BLOK PARAMETROW
# ============================================================

PARAMS = {
    # --- Parametry floty i przebiegu ---
    "calkowita_flota_pojazdow_pl": 21_077_000,
    "sredni_roczny_przebieg_km": (18608 + 17925) / 2,  # = 18 266.5
    "zuzycie_bev_kwh_100km": 21.0,
    "phev_wspolczynnik_wzgledem_bev": 0.30,
    "phev_korekta_niedoszacowania": 146711 / 81667,
    "udzial_ladowania_publicznego_dom_jednorodzinny": 0.10,
    "udzial_ladowania_publicznego_blok": 0.80,
    "korytarz_energia_na_sesje_kwh": 23.2,
    "docelowa_energia_na_sesje_kwh": 13.1,
    "iea_udzial_szybkiego_ladowania_publicznego": 0.10,
    "sila_wlasnej_lokalizacji": 1.0,
    "waga_luki_infrastrukturalnej": 0.5,
}

print("=== PARAMETRY MODELU (v2 - dane zamiast zalozen, gdzie to mozliwe) ===")
for k, v in PARAMS.items():
    print(f"  {k}: {v}")


def main():
    df = pd.read_csv("../data/candidate_locations_FINAL_v2.csv", low_memory=False)

    # UWAGA: kilka nazw powiatow powtarza sie w dwoch roznych wojewodztwach
    # (np. "brzeski" istnieje i w malopolskim, i w opolskim) - do grupowania
    # uzywamy wiec klucza zlozonego (wojewodztwo+nazwa), nie samej nazwy,
    # zeby ich nie mylic
    df["_powiat_klucz"] = df["powiat_wojewodztwo"].astype(str) + "|" + df["powiat_nazwa"].astype(str)

    bev_total = df.drop_duplicates(subset="_powiat_klucz")["powiat_liczba_bev"].sum()
    phev_total = (
        df.drop_duplicates(subset="_powiat_klucz")["powiat_liczba_phev"].sum()
        * PARAMS["phev_korekta_niedoszacowania"]
    )
    ev_total = bev_total + phev_total
    udzial_ev_floty = ev_total / PARAMS["calkowita_flota_pojazdow_pl"]
    print(f"\nSzacowany udzial EV w calej flocie PL: {udzial_ev_floty*100:.2f}%")

    # ---------- KALIBRACJA: calkowita energia EV w Polsce (top-down) ----------
    energia_bev_total = bev_total * PARAMS["sredni_roczny_przebieg_km"] * PARAMS["zuzycie_bev_kwh_100km"] / 100
    energia_phev_total = (
        phev_total * PARAMS["sredni_roczny_przebieg_km"] * PARAMS["zuzycie_bev_kwh_100km"] / 100
        * PARAMS["phev_wspolczynnik_wzgledem_bev"]
    )
    energia_ev_total_kwh = energia_bev_total + energia_phev_total
    cel_korytarz_kwh = energia_ev_total_kwh * PARAMS["iea_udzial_szybkiego_ladowania_publicznego"]
    print(f"Calkowita energia EV w Polsce (top-down): {energia_ev_total_kwh/1e6:.1f} GWh/rok")
    print(f"Cel dla segmentu korytarzowego (IEA {PARAMS['iea_udzial_szybkiego_ladowania_publicznego']*100:.0f}%): "
          f"{cel_korytarz_kwh/1e6:.1f} GWh/rok")

    # ---------- SEGMENT KORYTARZOWY ----------
    kor = df["segment"] == "korytarzowa"
    ev_ruch_roczny = df.loc[kor, "traffic_primary_sam_osobowe"] * udzial_ev_floty * 365
    # Konkurencja wazona MOCA (kW), nie sama liczba stacji - stacja 400kW
    # to realnie wieksze zagrozenie konkurencyjne niz stara stacja 22kW AC,
    # a przy liczeniu samych stacji liczylyby sie identycznie. Normalizacja
    # przez mediane mocy pojedynczej stacji w kraju (44 kW), zeby skala
    # pozostala porownywalna z poprzednim podejsciem (liczba stacji).
    MOC_TYPOWEJ_STACJI_KW = 44.0
    konkurencja = df.loc[kor, "existing_eipa_power_kw_active_2km"].fillna(0) / MOC_TYPOWEJ_STACJI_KW
    udzial_lokalizacji_kor = PARAMS["sila_wlasnej_lokalizacji"] / (PARAMS["sila_wlasnej_lokalizacji"] + konkurencja)

    # KALIBRACJA wspolczynnika zatrzymania: rozwiazujemy rownanie tak,
    # zeby suma energii korytarzowej = cel_korytarz_kwh (zamiast wpisywac
    # zalozona z gory wartosc)
    suma_bazowa = (ev_ruch_roczny * udzial_lokalizacji_kor * PARAMS["korytarz_energia_na_sesje_kwh"]).sum()
    wspolczynnik_zatrzymania_skalibrowany = cel_korytarz_kwh / suma_bazowa
    print(f"Skalibrowany wspolczynnik zatrzymania na korytarzu: "
          f"{wspolczynnik_zatrzymania_skalibrowany*100:.3f}% (poprzednio zalozone: 3%)")

    sesje_kor = ev_ruch_roczny * wspolczynnik_zatrzymania_skalibrowany * udzial_lokalizacji_kor
    energia_kor = sesje_kor * PARAMS["korytarz_energia_na_sesje_kwh"]

    df.loc[kor, "sesje_rocznie_szacunek"] = sesje_kor
    df.loc[kor, "energia_kwh_rocznie_szacunek"] = energia_kor
    df.loc[kor, "udzial_lokalizacji"] = udzial_lokalizacji_kor

    # ---------- SEGMENT DOCELOWY ----------
    dest = df["segment"] == "docelowa"

    # Sklonnosc do ladowania publicznego = wazona srednia obu wskaznikow
    # PSPA (dom jednorodzinny / blok), wazona RZECZYWISTYM udzialem
    # budynkow jednorodzinnych w powiecie (NSP 2021, GUS/BDL) - zamiast
    # poprzedniego proxy przez gestosc zaludnienia.
    udzial_jednorodzinne = df.loc[dest, "udzial_jednorodzinne_nsp"]
    sklonnosc_publiczna = (
        PARAMS["udzial_ladowania_publicznego_dom_jednorodzinny"] * udzial_jednorodzinne
        + PARAMS["udzial_ladowania_publicznego_blok"] * (1 - udzial_jednorodzinne)
    )

    # ---------- LUKA INFRASTRUKTURALNA POWIATU ----------
    # Dla kazdego powiatu (segment docelowy): czy ma proporcjonalnie
    # wiecej/mniej mocy EIPA niz sugerowalaby jego gestosc EV.
    powiat_baza = df.loc[dest].drop_duplicates(subset="_powiat_klucz")[
        ["_powiat_klucz", "powiat_ev_na_1000_mieszkancow", "powiat_moc_stacji_eipa_kw", "powiat_ludnosc"]
    ].copy()
    # NaN w mocy EIPA oznacza powiaty z zerowa zarejestrowana infrastruktura
    # (brak dopasowania w polaczonym pliku), nie brak danych - traktujemy jako 0
    powiat_baza["powiat_moc_stacji_eipa_kw"] = powiat_baza["powiat_moc_stacji_eipa_kw"].fillna(0)
    powiat_baza["moc_na_1000_mieszkancow"] = (
        powiat_baza["powiat_moc_stacji_eipa_kw"] / powiat_baza["powiat_ludnosc"] * 1000
    )
    powiat_baza["percentyl_ev"] = powiat_baza["powiat_ev_na_1000_mieszkancow"].rank(pct=True)
    powiat_baza["percentyl_infra"] = powiat_baza["moc_na_1000_mieszkancow"].rank(pct=True)
    powiat_baza["luka_infrastrukturalna"] = powiat_baza["percentyl_ev"] - powiat_baza["percentyl_infra"]

    luka_mapa = powiat_baza.set_index("_powiat_klucz")["luka_infrastrukturalna"]
    luka_lokalizacji = df.loc[dest, "_powiat_klucz"].map(luka_mapa)
    mnoznik_luki = 1 + PARAMS["waga_luki_infrastrukturalnej"] * luka_lokalizacji
    print(f"\nMnoznik luki infrastrukturalnej - zakres: {mnoznik_luki.min():.2f}x do {mnoznik_luki.max():.2f}x")
    print(f"Brakujace (NaN) po mapowaniu: {luka_lokalizacji.isna().sum()}")

    bev_lokalne = df.loc[dest, "powiat_liczba_bev"]
    phev_lokalne = df.loc[dest, "powiat_liczba_phev"] * PARAMS["phev_korekta_niedoszacowania"]
    dest_unique_mask = (df["segment"] == "docelowa") & (df["dedup_status"] == "unique")
    liczba_lokalizacji_w_powiecie = df["_powiat_klucz"].map(
        df[dest_unique_mask].groupby("_powiat_klucz").size()
    )

    energia_calkowita_bev = bev_lokalne * PARAMS["sredni_roczny_przebieg_km"] * PARAMS["zuzycie_bev_kwh_100km"] / 100
    energia_calkowita_phev = (
        phev_lokalne * PARAMS["sredni_roczny_przebieg_km"] * PARAMS["zuzycie_bev_kwh_100km"] / 100
        * PARAMS["phev_wspolczynnik_wzgledem_bev"]
    )
    energia_publiczna_powiat = (energia_calkowita_bev + energia_calkowita_phev) * sklonnosc_publiczna

    konkurencja_dest = df.loc[dest, "existing_eipa_power_kw_active_2km"].fillna(0) / MOC_TYPOWEJ_STACJI_KW
    udzial_lokalizacji_dest = PARAMS["sila_wlasnej_lokalizacji"] / (PARAMS["sila_wlasnej_lokalizacji"] + konkurencja_dest)

    energia_dest = (
        energia_publiczna_powiat / liczba_lokalizacji_w_powiecie.loc[dest]
        * udzial_lokalizacji_dest
        * mnoznik_luki
    )
    sesje_dest = energia_dest / PARAMS["docelowa_energia_na_sesje_kwh"]

    df.loc[dest, "sesje_rocznie_szacunek"] = sesje_dest
    df.loc[dest, "energia_kwh_rocznie_szacunek"] = energia_dest
    df.loc[dest, "udzial_lokalizacji"] = udzial_lokalizacji_dest
    df.loc[dest, "sklonnosc_ladowania_publicznego"] = sklonnosc_publiczna
    df.loc[dest, "luka_infrastrukturalna_powiatu"] = luka_lokalizacji
    df.loc[dest, "mnoznik_luki_infrastrukturalnej"] = mnoznik_luki

    # ---------- RANKING WZGLEDNY (nie prognoza bezwzgledna) ----------
    # Wyniki tego modelu nalezy interpretowac przede wszystkim jako ranking
    # PORZADKUJACY lokalizacje wzgledem siebie, nie jako precyzyjna
    # prognoze finansowa w kWh/rok (zbyt wiele parametrow to nadal
    # zalozenia/kalibracje, patrz sprawozdanie metodyczne). Dodajemy wiec
    # ranga percentylowa OSOBNO dla kazdego segmentu, zeby to podejscie
    # bylo widoczne wprost w danych, a nie tylko w dokumentacji.
    df.loc[kor, "ranga_percentylowa_w_segmencie"] = df.loc[kor, "energia_kwh_rocznie_szacunek"].rank(pct=True)
    df.loc[dest, "ranga_percentylowa_w_segmencie"] = df.loc[dest, "energia_kwh_rocznie_szacunek"].rank(pct=True)

    return df


if __name__ == "__main__":
    df = main()
    # MODYFIKACJA ŚCIEŻKI: Zapis do katalogu data/
    df.to_csv("../data/candidate_locations_z_popytem.csv", index=False)

    print("\n=== WYNIKI - PODSUMOWANIE ===")
    for seg in ["korytarzowa", "docelowa"]:
        sub = df[df["segment"] == seg]
        print(f"\n{seg} (n={len(sub)}):")
        print(f"  sesje/rok: min={sub['sesje_rocznie_szacunek'].min():.0f}, "
              f"mediana={sub['sesje_rocznie_szacunek'].median():.0f}, "
              f"p95={sub['sesje_rocznie_szacunek'].quantile(0.95):.0f}, "
              f"max={sub['sesje_rocznie_szacunek'].max():.0f}")
        print(f"  energia MWh/rok: mediana={sub['energia_kwh_rocznie_szacunek'].median()/1000:.1f}, "
              f"p95={sub['energia_kwh_rocznie_szacunek'].quantile(0.95)/1000:.1f}, "
              f"max={sub['energia_kwh_rocznie_szacunek'].max()/1000:.1f}")

    print(f"\nSUMA energii/rok (caly kraj, wszystkie kandydackie lokalizacje): "
          f"{df['energia_kwh_rocznie_szacunek'].sum()/1_000_000:.1f} GWh")

=== PARAMETRY MODELU (v2 - dane zamiast zalozen, gdzie to mozliwe) ===
  calkowita_flota_pojazdow_pl: 21077000
  sredni_roczny_przebieg_km: 18266.5
  zuzycie_bev_kwh_100km: 21.0
  phev_wspolczynnik_wzgledem_bev: 0.3
  phev_korekta_niedoszacowania: 1.7964538920249304
  udzial_ladowania_publicznego_dom_jednorodzinny: 0.1
  udzial_ladowania_publicznego_blok: 0.8
  korytarz_energia_na_sesje_kwh: 23.2
  docelowa_energia_na_sesje_kwh: 13.1
  iea_udzial_szybkiego_ladowania_publicznego: 0.1
  sila_wlasnej_lokalizacji: 1.0
  waga_luki_infrastrukturalnej: 0.5

Szacowany udzial EV w calej flocie PL: 1.49%
Calkowita energia EV w Polsce (top-down): 810.4 GWh/rok
Cel dla segmentu korytarzowego (IEA 10%): 81.0 GWh/rok
Skalibrowany wspolczynnik zatrzymania na korytarzu: 2.262% (poprzednio zalozone: 3%)

Mnoznik luki infrastrukturalnej - zakres: 0.52x do 1.39x
Brakujace (NaN) po mapowaniu: 0

=== WYNIKI - PODSUMOWANIE ===

korytarzowa (n=3564):
  sesje/rok: min=2, mediana=465, p95=3719, max=10366
  ene